In [1]:
import os
import pandas as pd
from pathlib import Path
import shutil

In [2]:
os.chdir('C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\csv')
os.listdir()

['calc_case_description_test_set.csv',
 'calc_case_description_train_set.csv',
 'dicom_info.csv',
 'mass_case_description_test_set.csv',
 'mass_case_description_train_set.csv',
 'meta.csv']

In [3]:
train_data = pd.read_csv('C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\csv\\mass_case_description_train_set.csv')
test_data = pd.read_csv('C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\csv\\mass_case_description_test_set.csv')

In [4]:
train_data.columns

Index(['patient_id', 'breast_density', 'left or right breast', 'image view',
       'abnormality id', 'abnormality type', 'mass shape', 'mass margins',
       'assessment', 'pathology', 'subtlety', 'image file path',
       'cropped image file path', 'ROI mask file path'],
      dtype='object')

### Extract Series UID

In [5]:

def extract_uid(path):
    if pd.isna(path):
        return None
    return str(path).split('/')[-2]

# Extract UID
train_uid = train_data['image file path'].apply(extract_uid)
test_uid = test_data['image file path'].apply(extract_uid)

# Insert as first column
train_data.insert(0, 'SeriesUID', train_uid)
test_data.insert(0, 'SeriesUID', test_uid)

# Check


In [6]:
print(train_data[['SeriesUID']].head())
print(test_data[['SeriesUID']].head())

                                           SeriesUID
0  1.3.6.1.4.1.9590.100.1.2.342386194811267636608...
1  1.3.6.1.4.1.9590.100.1.2.359308329312397897125...
2  1.3.6.1.4.1.9590.100.1.2.891800462110225318343...
3  1.3.6.1.4.1.9590.100.1.2.295360926313492745441...
4  1.3.6.1.4.1.9590.100.1.2.410524754913057908920...
                                           SeriesUID
0  1.3.6.1.4.1.9590.100.1.2.245063149211255120613...
1  1.3.6.1.4.1.9590.100.1.2.859522146111705060178...
2  1.3.6.1.4.1.9590.100.1.2.221311896128932948279...
3  1.3.6.1.4.1.9590.100.1.2.239949064412092068706...
4  1.3.6.1.4.1.9590.100.1.2.215081818713600536113...


### Merge Train and Test data first

As, It is not Tabular data, no scaling or imputation has been done so, there is no risk of data Leakage. 

In [7]:
all_data = pd.concat([train_data, test_data], ignore_index=True)

In [8]:
all_data['pathology'].value_counts()

pathology
MALIGNANT                  784
BENIGN                     771
BENIGN_WITHOUT_CALLBACK    141
Name: count, dtype: int64

In [9]:
all_data_clean = all_data[all_data['pathology'] != 'BENIGN_WITHOUT_CALLBACK' ].copy()

In [10]:
all_data_clean.shape

(1555, 15)

In [11]:
jpeg_root = Path(r"C:\Users\moham\Downloads\archive\jpeg")

uids = set(all_data_clean['SeriesUID'].astype(str))

jpeg_folders = {folder.name for folder in jpeg_root.iterdir() if folder.is_dir()}

matched_uids = uids.intersection(jpeg_folders)

print("UIDs in dataframe:", len(uids))
print("JPEG folders:", len(jpeg_folders))
print("Matches:", len(matched_uids))
print("Missing:", len(uids) - len(matched_uids))

UIDs in dataframe: 1473
JPEG folders: 6774
Matches: 1473
Missing: 0


In [12]:
jpeg_root = Path(r"C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\jpeg")
target_root = Path(r"C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\jpeg_selected")

target_root.mkdir(exist_ok=True)

for uid in matched_uids:
    src = jpeg_root / uid
    dst = target_root / uid

    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)

print(f"Copied {len(matched_uids)} folders.")

Copied 1473 folders.


In [13]:
all_data_final = all_data_clean[
    all_data_clean['SeriesUID'].isin(matched_uids)
].copy()

print(all_data_final.shape)
print(all_data_final['pathology'].value_counts())

(1555, 15)
pathology
MALIGNANT    784
BENIGN       771
Name: count, dtype: int64


In [14]:
print(all_data_final['SeriesUID'].nunique())

1473


### Duplicate Rows Issue 

All Series UID have matached but still Rows in parent file are 1555 suggesting there are 82 duplicates. Let's explore them first and then decide how we will do it? 

In [15]:
print(len(matched_uids))
print(all_data_clean['SeriesUID'].nunique())

1473
1473


In [16]:
all_data_final.shape

(1555, 15)

In [17]:
uid_counts = all_data_clean['SeriesUID'].value_counts()

duplicate_uids = uid_counts[uid_counts > 1]

print("Duplicate UIDs:", len(duplicate_uids))
print(duplicate_uids.head(5))

Duplicate UIDs: 59
SeriesUID
1.3.6.1.4.1.9590.100.1.2.87251504411596839017815563663575708222     6
1.3.6.1.4.1.9590.100.1.2.354587724213018641829708719832963731890    5
1.3.6.1.4.1.9590.100.1.2.101999469712679926627011488331183444331    4
1.3.6.1.4.1.9590.100.1.2.292605978712963936606864280561587921668    4
1.3.6.1.4.1.9590.100.1.2.122014842013197980401803659680241641105    3
Name: count, dtype: int64


In [18]:
dup_rows = all_data_clean[
    all_data_clean['SeriesUID'].isin(duplicate_uids.index)
].sort_values('SeriesUID')

dup_rows.head(5)

,SeriesUID,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
26,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,1,mass,OVAL,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_1/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_1/1.3.6.1.4.1.9...
27,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,2,mass,OVAL,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_2/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_2/1.3.6.1.4.1.9...
28,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,3,mass,OVAL,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_3/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_3/1.3.6.1.4.1.9...
29,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,4,mass,LOBULATED,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_4/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_4/1.3.6.1.4.1.9...
310,1.3.6.1.4.1.9590.100.1.2.108388395511926748633...,P_00432,1,LEFT,MLO,2,mass,IRREGULAR,SPICULATED,4,MALIGNANT,3,Mass-Training_P_00432_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00432_LEFT_MLO_2/1.3.6.1.4.1.9...,Mass-Training_P_00432_LEFT_MLO_2/1.3.6.1.4.1.9...


The duplicates are for the same patient ID and are depending on the abnormalities. As our task is to classify Benign from Malignant, that's whu we will drop the duplicates keeping only one input to Target. 

In [19]:
all_data_image = all_data_clean.drop_duplicates(
    subset=['SeriesUID']
).copy()

In [20]:
all_data_image.shape

(1473, 15)

Image Processing
----

In [29]:
print("Unique patients:", all_data_final['patient_id'].nunique())

Unique patients: 822


In [30]:
all_data_final.groupby('patient_id')['pathology'].nunique()

patient_id
P_00001    1
P_00004    1
P_00009    1
P_00015    1
P_00016    1
          ..
P_01981    1
P_01983    1
P_02033    1
P_02079    1
P_02092    1
Name: pathology, Length: 822, dtype: int64

In [31]:
all_data_final.groupby('patient_id').size()

patient_id
P_00001    2
P_00004    3
P_00009    2
P_00015    1
P_00016    2
          ..
P_01981    2
P_01983    1
P_02033    2
P_02079    2
P_02092    2
Length: 822, dtype: int64

In [32]:
patient_pathology = all_data_final.groupby('patient_id')['pathology'].nunique()

print("Patients with >1 pathology:")
print((patient_pathology > 1).sum())

Patients with >1 pathology:
14


In [35]:
patient_pathology = all_data_final.groupby('patient_id')['pathology'].nunique()

mixed_patients = patient_pathology[patient_pathology > 1].index

for patient_id in mixed_patients:
    print(f"\nPatient ID: {patient_id}")
    print(
        all_data_final[
            all_data_final['patient_id'] == patient_id
        ][['SeriesUID', 'pathology']]
    )


Patient ID: P_00160
                                             SeriesUID  pathology
118  1.3.6.1.4.1.9590.100.1.2.192830829713156688526...  MALIGNANT
119  1.3.6.1.4.1.9590.100.1.2.305474508013164328209...  MALIGNANT
120  1.3.6.1.4.1.9590.100.1.2.103311533911270368932...     BENIGN
121  1.3.6.1.4.1.9590.100.1.2.340775804311000418009...     BENIGN

Patient ID: P_00419
                                             SeriesUID  pathology
286  1.3.6.1.4.1.9590.100.1.2.175043772610570804328...     BENIGN
287  1.3.6.1.4.1.9590.100.1.2.175043772610570804328...  MALIGNANT
288  1.3.6.1.4.1.9590.100.1.2.727246052124129251237...  MALIGNANT
289  1.3.6.1.4.1.9590.100.1.2.727246052124129251237...     BENIGN
290  1.3.6.1.4.1.9590.100.1.2.243518844011228052739...     BENIGN
291  1.3.6.1.4.1.9590.100.1.2.639840744110942940280...     BENIGN

Patient ID: P_00596
                                             SeriesUID  pathology
418  1.3.6.1.4.1.9590.100.1.2.184457361110630942322...     BENIGN
419  1.3.6.1.

In [36]:
mixed_patients = patient_pathology[patient_pathology > 1].index

for patient_id in mixed_patients:
    print("\n" + "="*80)
    print("Patient:", patient_id)

    display(
        all_data_final[
            all_data_final['patient_id'] == patient_id
        ][[
            'SeriesUID',
            'left or right breast',
            'image view',
            'abnormality id',
            'pathology'
        ]]
    )


Patient: P_00160


,SeriesUID,left or right breast,image view,abnormality id,pathology
118,1.3.6.1.4.1.9590.100.1.2.192830829713156688526...,LEFT,CC,1,MALIGNANT
119,1.3.6.1.4.1.9590.100.1.2.305474508013164328209...,LEFT,MLO,1,MALIGNANT
120,1.3.6.1.4.1.9590.100.1.2.103311533911270368932...,RIGHT,CC,1,BENIGN
121,1.3.6.1.4.1.9590.100.1.2.340775804311000418009...,RIGHT,MLO,1,BENIGN



Patient: P_00419


,SeriesUID,left or right breast,image view,abnormality id,pathology
286,1.3.6.1.4.1.9590.100.1.2.175043772610570804328...,LEFT,CC,1,BENIGN
287,1.3.6.1.4.1.9590.100.1.2.175043772610570804328...,LEFT,CC,2,MALIGNANT
288,1.3.6.1.4.1.9590.100.1.2.727246052124129251237...,LEFT,MLO,1,MALIGNANT
289,1.3.6.1.4.1.9590.100.1.2.727246052124129251237...,LEFT,MLO,2,BENIGN
290,1.3.6.1.4.1.9590.100.1.2.243518844011228052739...,RIGHT,CC,1,BENIGN
291,1.3.6.1.4.1.9590.100.1.2.639840744110942940280...,RIGHT,MLO,1,BENIGN



Patient: P_00596


...

,SeriesUID,left or right breast,image view,abnormality id,pathology
1646,1.3.6.1.4.1.9590.100.1.2.406152385910425726329...,RIGHT,CC,1,MALIGNANT
1647,1.3.6.1.4.1.9590.100.1.2.462829403104100447070...,RIGHT,MLO,1,BENIGN



Patient: P_01644


,SeriesUID,left or right breast,image view,abnormality id,pathology
1142,1.3.6.1.4.1.9590.100.1.2.689976205122366323175...,LEFT,CC,1,BENIGN
1143,1.3.6.1.4.1.9590.100.1.2.373282707011960449216...,LEFT,MLO,1,BENIGN
1144,1.3.6.1.4.1.9590.100.1.2.426585149212376352035...,RIGHT,CC,1,MALIGNANT
1145,1.3.6.1.4.1.9590.100.1.2.150586274112163348618...,RIGHT,MLO,1,MALIGNANT



Patient: P_01800


,SeriesUID,left or right breast,image view,abnormality id,pathology
1684,1.3.6.1.4.1.9590.100.1.2.159593717111637608173...,LEFT,CC,1,MALIGNANT
1685,1.3.6.1.4.1.9590.100.1.2.296793691211243286711...,LEFT,MLO,1,MALIGNANT
1686,1.3.6.1.4.1.9590.100.1.2.257796728813960468725...,RIGHT,CC,1,BENIGN


In [39]:
breast_pathology = (
    all_data_final
    .groupby(['patient_id', 'left or right breast'])['pathology']
    .nunique()
)

conflicting_breasts = breast_pathology[
    breast_pathology > 1
]

print("Conflicting breasts:", len(conflicting_breasts))

Conflicting breasts: 5


In [43]:
conflicting_pairs = set(conflicting_breasts.index)

mask = all_data_image.apply(
    lambda row: (
        row['patient_id'],
        row['left or right breast']
    ) not in conflicting_pairs,
    axis=1
)

all_data_filtered = all_data_image[mask].copy()

In [44]:
all_data_filtered.shape

(1463, 15)

In [45]:
keep_uids = set(all_data_filtered['SeriesUID'].astype(str))

deleted = 0

for folder in target_root.iterdir():

    if folder.is_dir():

        if folder.name not in keep_uids:

            shutil.rmtree(folder)
            deleted += 1

print(f"Deleted {deleted} folders")
print(f"Remaining folders should be {len(keep_uids)}")

Deleted 10 folders
Remaining folders should be 1463


In [46]:
all_data_filtered.groupby('patient_id').size().describe()

count    819.000000
mean       1.786325
std        0.565520
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max        4.000000
dtype: float64

In [47]:
all_data_filtered['patient_id'].value_counts().head(20)

patient_id
P_00711    4
P_00820    4
P_00224    4
P_00710    4
P_00021    4
P_00596    4
P_00332    4
P_00160    4
P_00303    4
P_00391    4
P_00173    4
P_01270    4
P_01394    4
P_01644    4
P_01712    4
P_00092    4
P_01739    4
P_00515    3
P_01090    3
P_00004    3
Name: count, dtype: int64

In [48]:
from sklearn.model_selection import train_test_split
from pathlib import Path

patients = all_data_filtered['patient_id'].unique()

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.2,
    random_state=42,
    stratify=all_data_filtered.groupby('patient_id')['pathology'].first()
)


train_df = all_data_filtered[
    all_data_filtered['patient_id'].isin(train_patients)
].copy()

test_df = all_data_filtered[
    all_data_filtered['patient_id'].isin(test_patients)
].copy()

y_train = (train_df['pathology'] == 'MALIGNANT').astype(int)
y_test = (test_df['pathology'] == 'MALIGNANT').astype(int)


X_train = [
    str(target_root / uid)
    for uid in train_df['SeriesUID']
]

X_test = [
    str(target_root / uid)
    for uid in test_df['SeriesUID']
]

print("Train patients:", len(train_patients))
print("Test patients:", len(test_patients))

print("Train images:", len(X_train))
print("Test images:", len(X_test))

print("Train malignant:", y_train.sum())
print("Test malignant:", y_test.sum())

Train patients: 655
Test patients: 164
Train images: 1164
Test images: 299
Train malignant: 589
Test malignant: 149
